<a href="https://colab.research.google.com/github/franfgv9/PLN/blob/main/Practica_6_Transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers[torch]
!pip install sentence-transformers

In [ ]:
texto = """
A Universidade de Coimbra (UC) é uma universidade pública localizada
na cidade de Coimbra , em Portugal. É uma das universidades mais
antigas do mundo ainda em operação, sendo a mais antiga e uma das
maiores do país. Composta por 3 polos , 8 faculdades e 18 museus , a
instituição conta ainda com o Jardim Botânico e o Estádio
Universitário de Coimbra em um espaço com 25 188 alunos em 2020.
A sua história remonta ao século seguinte ao da fundação da nação
portuguesa , dado que foi criada a 1 de Março de 1290, quando o rei
D. Dinis I assinou na cidade de Leiria o documento Scientiae
thesaurus mirabilis , criando a universidade , o qual foi
intermediado e confirmado pelo Papa. Fixada definitivamente na
cidade de Coimbra em 1537, sete anos depois , todas as suas
faculdades se instalam no antigo Paço Real da Alcáçova (denominado
Paço das Escolas , após a sua aquisição pela Universidade de
Coimbra em 1597).
Organizada em oito faculdades , de acordo com uma variedade de campos
de conhecimento , a universidade oferece todos os graus académicos
em arquitetura , educação, engenharia , humanidades , direito , matemá
tica , medicina , ciências naturais , psicologia , ciências sociais e
desporto.

A Universidade de Coimbra possui aproximadamente 25 mil estudantes ,
abrangendo uma das maiores comunidades de estudantes
internacionais em Portugal , sendo a sua universidade mais
cosmopolita. Além disso , é o membro -criador do chamado Grupo
Coimbra , uma rede de universidades europeias cujo objetivo é a
colaboração académica entre os elementos do grupo. Em 22 de junho
de 2013 foi declarada Património Mundial pela Organização das Naçõ
es Unidas para a Educação, a Ciência e a Cultura (UNESCO).
"""

CONTEXTO:

GPT --> ideal para predecir la siguiente palabra (generación de texto)

BERT --> ideal para comprensión de texto

Tanto GloVe como Word2vec como GPT y BERT son modelos preenenados (transformers) para predecir palabras

#  Apartado 1.2 – Exemplos

Esta sección te enseña cómo usar los pipelines de Hugging Face, una forma muy sencilla de aplicar modelos preentrenados de PLN sin necesidad de definir arquitecturas ni entrenar redes desde cero.

## Exemplo 1.1 – BERTimbau (Previsão de tokens mascaradas)

🧠 Explicación teórica

El modelo BERTimbau es una versión de BERT (Bidirectional Encoder Representations from Transformers) entrenada específicamente para portugués, desarrollada por NeuralMind.
Existen dos versiones:

neuralmind/bert-base-portuguese-cased

neuralmind/bert-large-portuguese-cased

Este modelo fue entrenado para comprender el contexto bidireccional del texto.
Una de sus tareas principales es la previsão de tokens mascaradas (Masked Language Modeling), que consiste en predecir palabras que se han reemplazado por una máscara [MASK]

El pipeline que se utiliza en este caso es:

pipeline("fill-mask")


👉 Este pipeline rellena el espacio enmascarado con las palabras más probables según el contexto.

In [ ]:
# Importar o pipeline da biblioteca transformers
from transformers import pipeline

# Frase com uma palavra mascarada
frase = "A próxima [MASK] está mascarada."

# Criar o pipeline de Masked Language Modeling com o modelo BERTimbau base
mlm = pipeline("fill-mask", model='neuralmind/bert-base-portuguese-cased')

# Obter as 10 palavras mais prováveis para substituir [MASK]
candidatos = mlm(frase, top_k=10)

# Imprimir resultados
for c in candidatos:
    print(c['token_str'], c['score'])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/647 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Some weights of the model checkpoint at neuralmind/bert-base-portuguese-cased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/43.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Device set to use cuda:0


geração 0.1356668919324875
versão 0.040750131011009216
vítima 0.03570321574807167
etapa 0.03370468318462372
vez 0.026194993406534195
guerra 0.024902287870645523
fase 0.023033026605844498
já 0.01950095221400261
eleição 0.017349136993288994
onda 0.01645560935139656


## Exemplo 1.2 – Perguntas e Respostas (Question Answering)
Explicación teórica

En este ejemplo seguimos usando el modelo BERTimbau, pero una versión fine-tuned (afinada) específicamente para la tarea de pregunta–respuesta sobre texto.
Esta tarea pertenece a lo que se conoce como Reading Comprehension, donde el modelo debe leer un contexto y extraer directamente del texto la respuesta a una pregunta.

El modelo que se utiliza es:

pierreguillou/bert-base-cased-squad-v1.1-portuguese


Este modelo fue entrenado en una versión traducida al portugués del famoso conjunto de datos SQuAD (Stanford Question Answering Dataset).

👉 La pipeline usada es:

pipeline("question-answering")

Este pipeline recibe dos argumentos principales:

question: la pregunta formulada en lenguaje natural.

context: el texto sobre el que se debe buscar la respuesta.

El modelo devuelve:

answer: la respuesta textual.

score: la confianza del modelo (valor entre 0 y 1).

In [ ]:
# Importar o pipeline da biblioteca transformers
from transformers import pipeline

# Pergunta de exemplo  -->  se basa en la info del texto definido anteriormente
pergunta = "Onde fica a Universidade de Coimbra?"

# Criar o pipeline de Perguntas e Respostas
qa = pipeline("question-answering",
              model='pierreguillou/bert-base-cased-squad-v1.1-portuguese')

# Obter a resposta a partir do texto do exercício anterior
resposta = qa(question=pergunta, context=texto)

# Mostrar resposta e grau de confiança
print("Resposta:", resposta['answer'])
print("Confiança:", resposta['score'])

config.json:   0%|          | 0.00/862 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/433M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/494 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Device set to use cuda:0


Resposta: cidade de Coimbra , em Portugal
Confiança: 0.37238386273384094


In [ ]:
# Importar o pipeline da biblioteca transformers
from transformers import pipeline

# Pergunta de exemplo  -->  se basa en la info del texto definido anteriormente
pergunta = "Quantos polos têm a Universidade de Coimbra?"

# Criar o pipeline de Perguntas e Respostas
qa = pipeline("question-answering",
              model='pierreguillou/bert-base-cased-squad-v1.1-portuguese')

# Obter a resposta a partir do texto do exercício anterior
resposta = qa(question=pergunta, context=texto)

# Mostrar resposta e grau de confiança
print("Resposta:", resposta['answer'])
print("Confiança:", resposta['score'])

Device set to use cuda:0


Resposta: 3
Confiança: 0.9099551439285278


## Exemplo 1.3 – Sumarização de Texto com T5 (Text Summarization)
Explicación teórica

La sumarización automática consiste en generar una versión reducida de un texto largo, manteniendo las ideas principales.
El modelo utilizado en este ejemplo es una versión del T5 (Text-to-Text Transfer Transformer) afinada para el portugués:

recogna-nlp/ptt5-base-summ-cstnews


Este modelo fue entrenado específicamente en la tarea de resumir noticias en portugués (corpus CSTNews).
El pipeline empleado es:

pipeline("summarization")


Este pipeline requiere una librería adicional llamada SentencePiece, necesaria para manejar la tokenización del modelo T5.

In [ ]:
!pip install sentencepiece

El pipeline acepta parámetros como:

min_new_tokens: número mínimo de tokens generados en el resumen.

max_new_tokens: número máximo de tokens generados.

In [ ]:
# Importar o pipeline
from transformers import pipeline

# Criar o pipeline de sumarização com o modelo T5 em português
summarizer = pipeline('summarization', model='recogna-nlp/ptt5-base-summ-cstnews')

# Gerar um sumário do texto da Universidade de Coimbra
sumario = summarizer(texto, min_new_tokens=10, max_new_tokens=100)

# Mostrar o sumário
print(sumario[0]['summary_text'])

config.json:   0%|          | 0.00/669 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/756k [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
Device set to use cuda:0


A Universidade de Coimbra (UC) é uma universidade pública localizada na cidade de Coimbra , em Portugal. Composta por 3 polos, 8 faculdades e 18 museus, a universidade conta ainda com o Jardim Botânico e o Estádio Universitário de Coimbra em um espaço com 25 188 alunos em 2020. A sua história remonta ao século seguinte ao da fundação da nação portuguesa , dado que foi criada em 1290, quando o rei D. Dinis I assinou o documento Scientiae thesaurus mira


## Exemplo 1.4 – Geração de Texto com GPT-2 (Text Generation)
Explicación teórica

El modelo GPT-2 (Generative Pre-trained Transformer 2) fue diseñado para predecir la próxima palabra en una secuencia, dada la información previa.
En este ejemplo, se usa una versión afinada para el idioma portugués, creada por Pierre Guillou:

pierreguillou/gpt2-small-portuguese


Este modelo puede generar continuaciones coherentes de un texto inicial (prompt).
El pipeline empleado es:

pipeline("text-generation")


Los parámetros más importantes son:

prompt: texto inicial que el modelo completará.

max_new_tokens: número máximo de tokens (palabras o fragmentos) que se generarán.

temperature: controla la creatividad del modelo:

valores bajos (0.2–0.6) → texto más lógico y predecible;

valores altos (0.8–1.2) → texto más variado y creativo.

num_return_sequences: número de versiones diferentes que generará el modelo.

In [ ]:
# Importar o pipeline da biblioteca transformers
from transformers import pipeline

# Frase inicial (prompt) para a geração de texto
prompt = "A Universidade de Coimbra é"

# Criar o pipeline de geração de texto com GPT-2 em português
generator = pipeline("text-generation", model='pierreguillou/gpt2-small-portuguese')

# Gerar três sequências alternativas com temperatura média
resultado = generator(
    prompt,
    max_new_tokens=25,
    pad_token_id=50256,
    temperature=0.6,
    num_return_sequences=3
)

# Mostrar os textos gerados
for i, r in enumerate(resultado, 1):
    print(f"\n--- Texto {i} ---")
    print(r['generated_text'])

config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/510M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/510M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/92.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/120 [00:00<?, ?B/s]

Device set to use cuda:0



--- Texto 1 ---
A Universidade de Coimbra é uma instituição de ensino superior privada, com um campus na cidade de Coimbra, Portugal.

Foi criada a 5 de Abril

--- Texto 2 ---
A Universidade de Coimbra é uma das mais importantes instituições de ensino superior do país, tendo sido criada em 1969, tendo sido a instituição mais antiga da União

--- Texto 3 ---
A Universidade de Coimbra é um dos centros culturais da Universidade de Coimbra e da Região Autónoma dos Açores, com funções em Portugal, na Região Autónoma dos Açores


## Exemplo 1.5 – Extração de Features com Transformers
Explicación teórica

Hasta ahora, hemos usado pipelines preparados para tareas específicas (fill-mask, question-answering, summarization, text-generation).
Sin embargo, los modelos Transformers también pueden ser usados como extractores de características (feature extractors).

En este caso, no le pedimos al modelo que “responda” o “resuma”, sino que nos devuelva los vectores internos (embeddings) que representan el significado de cada token.
Estos vectores pueden servir para tareas como:

Clustering de textos

Cálculo de similitud semántica

Alimentar otros modelos de Machine Learning

Para eso, usamos el pipeline:

pipeline("feature-extraction")


Cada salida será un tensor 3D con la forma:

(batch_size, número_de_tokens, dimensão_oculta)


Por ejemplo, (1, 8, 768) para un modelo BERT base.

👉 En este ejemplo se utiliza BERTimbau nuevamente:

neuralmind/bert-base-portuguese-cased

In [ ]:
# Importar as bibliotecas necessárias
from transformers import pipeline
import numpy as np

# Criar o pipeline de extração de features com o BERTimbau
featurizer = pipeline('feature-extraction', model='neuralmind/bert-base-portuguese-cased')

# Texto de exemplo
texto_exemplo = "Isto é uma frase qualquer."

# Obter as features (vetores internos do modelo)
features = featurizer(texto_exemplo)

# Ver a forma do array retornado
print(np.array(features).shape)  # Exemplo: (1, 8, 768)  768 dimension de un vector embedding o de los 8?

Device set to use cuda:0


(1, 8, 768)


## Exemplo 1.6 – Sentence Transformers e Similaridade Semântica
🧠 Explicación teórica

Aunque podemos usar un modelo Transformer como BERTimbau para extraer embeddings, estos no siempre representan bien el significado global de una frase, ya que BERT fue entrenado principalmente para predecir palabras (tarea masked language modeling).

Para obtener vetores que representem o significado total da frase, utilizamos Sentence Transformers 🧩
Estos modelos están afinados en tarefas de NLI (Inferência Lógica Natural) ou STS (Semantic Textual Similarity), aprendiendo a mapear frases semelhantes para vetores próximos no espaço vetorial.

👉 En este ejemplo se usa:

ricardo-filho/bert-portuguese-cased-nli-assin-assin-2


Un modelo portugués entrenado con las colecciones ASSIN 1 y ASSIN 2, diseñadas para evaluar similitud textual y inferencia semántica.

La biblioteca utilizada es diferente:

from sentence_transformers import SentenceTransformer, util


El flujo general del ejemplo es:

Codificar un conjunto de frases en embeddings.

Calcular la similaridad coseno entre todas las posibles combinaciones de frases.

Analizar cuáles son más parecidas o más diferentes.

In [ ]:
# Importar as bibliotecas necessárias
from itertools import combinations
from sentence_transformers import SentenceTransformer, util

# Lista de frases para comparação
frases = [
    "Um garoto está fazendo um discurso",
    "Um garoto está falando",
    "Um homem não está tocando bateria",
    "Um homem está tocando o violão",
    "Uma mulher está andando a cavalo",
    "Alguma coisa está sendo frita por uma mulher"
]

# Carregar o modelo treinado para NLI/STS em português
model = SentenceTransformer('ricardo-filho/bert-portuguese-cased-nli-assin-assin-2')

# Obter os embeddings de todas as frases --> modelo preentenado para que saque los mejores enbeddings para conseguir una buena similitud de las palabras
embeddings = model.encode(frases)

# Calcular a similaridade coseno entre todos os pares de frases
for i, j in combinations(range(len(frases)), 2):
    sim = util.pytorch_cos_sim(embeddings[i], embeddings[j])
    print(f"{frases[i]}  <->  {frases[j]}:  Similaridade = {sim.item():.4f}")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/861 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Um garoto está fazendo um discurso  <->  Um garoto está falando:  Similaridade = 0.8523
Um garoto está fazendo um discurso  <->  Um homem não está tocando bateria:  Similaridade = -0.0197
Um garoto está fazendo um discurso  <->  Um homem está tocando o violão:  Similaridade = 0.8618
Um garoto está fazendo um discurso  <->  Uma mulher está andando a cavalo:  Similaridade = 0.8392
Um garoto está fazendo um discurso  <->  Alguma coisa está sendo frita por uma mulher:  Similaridade = 0.7388
Um garoto está falando  <->  Um homem não está tocando bateria:  Similaridade = 0.1174
Um garoto está falando  <->  Um homem está tocando o violão:  Similaridade = 0.8816
Um garoto está falando  <->  Uma mulher está andando a cavalo:  Similaridade = 0.7368
Um garoto está falando  <->  Alguma coisa está sendo frita por uma mulher:  Similaridade = 0.6683
Um homem não está tocando bateria  <->  Um homem está tocando o violão:  Similaridade = 0.0544
Um homem não está tocando bateria  <->  Uma mulher está an

## Exemplo 1.7 – Treino de um modelo Transformer para Classificação de Texto
🧠 Explicación teórica

Hasta ahora hemos usado modelos ya entrenados (pretrained) para tareas específicas.
En este ejemplo, veremos cómo reentrenar parcialmente un Transformer (ajustar sus pesos) para una nova tarefa supervisionada, llamada fine-tuning.

En este caso, la tarea es classificar títulos de jornal:

1 → título do jornal Público (notícia real)

0 → título do Inimigo Público (notícia humorística)

El modelo utilizado será Albertina-PTPT-100m, un Transformer do tipo DeBERTa, pré-treinado em português europeu.

### IMPORTANTE: este ejercicio es el verdaderamente importante para el proyecto

### a) Descargar os dados

In [ ]:
!wget https://raw.githubusercontent.com/NLP-CISUC/Recognizing-Humor-in-Portuguese/master/Datasets/Originais/TitulosPublico_N.txt -O publico.txt
!wget https://raw.githubusercontent.com/NLP-CISUC/Recognizing-Humor-in-Portuguese/master/Datasets/Originais/inimigopublico_H.txt -O ipublico.txt

--2025-11-11 17:23:40--  https://raw.githubusercontent.com/NLP-CISUC/Recognizing-Humor-in-Portuguese/master/Datasets/Originais/TitulosPublico_N.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 159919 (156K) [text/plain]
Saving to: ‘publico.txt’

publico.txt         100%[===================>] 156.17K  --.-KB/s    in 0.003s  

2025-11-11 17:23:40 (52.0 MB/s) - ‘publico.txt’ saved [159919/159919]

--2025-11-11 17:23:40--  https://raw.githubusercontent.com/NLP-CISUC/Recognizing-Humor-in-Portuguese/master/Datasets/Originais/inimigopublico_H.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... conne

### b) Carregar os dados e criar listas de texto e classes

In [ ]:
def carrega_docs(ficheiro, coment):
    docs = []
    with open(ficheiro) as f:
        lines = f.readlines()
        for txt in lines:
            txt = txt.strip()
            if len(txt) > 0 and txt[0] != coment:
                docs.append(txt)
    return docs

docs_p = carrega_docs('publico.txt', '#')
docs_ip = carrega_docs('ipublico.txt', '#')

# Criar listas de frases e rótulos (1 = Público, 0 = Inimigo Público)
lista_x = docs_p + docs_ip
lista_y = [1 if i < len(docs_p) else 0 for i in range(len(lista_x))]

### c) Separar os dados em treino, validação e teste

In [ ]:
from sklearn.model_selection import train_test_split

# Dividimos en 70% train 30% validacion + test
X_train, X_val_and_test, y_train, y_val_and_test = train_test_split(
    lista_x, lista_y, test_size=0.3, random_state=42)
# Dividimos validacion + test en 15% validación y 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_val_and_test, y_val_and_test, test_size=0.5, random_state=42)

### d) Tokenizar os textos

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("PORTULAN/albertina-100m-portuguese-ptpt-encoder")

train_tokenized = tokenizer(X_train, truncation=True, padding=True)     # añadimos padding
val_tokenized = tokenizer(X_val, truncation=True, padding=True)
test_tokenized = tokenizer(X_test, truncation=True, padding=True)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/963 [00:00<?, ?B/s]

### e) Criar o Dataset para o PyTorch

In [ ]:
import torch

class MyDataset(torch.utils.data.Dataset):    # Esta clase hereda de torch.utils.data.Dataset, que es la clase base de PyTorch para representar conjuntos de datos
# Porque así PyTorch (y HuggingFace Trainer) saben cómo: obtener un ejemplo por índice (__getitem__), saber cuántos ejemplos hay (__len__)
    def __init__(self, encodings, labels):      # Constructor
        self.encodings = encodings        # las salidas del tokenizer (diccionario con input_ids, attention_mask, etc.)
        self.labels = labels          # las etiquetas

    def __getitem__(self, idx):       # Define cómo obtener el ejemplo número idx del dataset --> Este método es llamado cuando hacemos algo como: train_dataset[0] → devuelve el primer ejemplo
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        #         item = {
        #     'input_ids': tensor(correspondiente_al_exemplo_idx),
        #     'attention_mask': tensor(correspondiente_al_exemplo_idx),
        #     ...
        # }

        # Aquí se añade una nueva clave al diccionario item: 'labels ' con la etiqueta correspondiente al ejemplo idx (0 o 1)
        item['labels'] = torch.tensor(self.labels[idx])
        return item
        # Devuelve el diccionario item, que contiene:
        # Las entradas del modelo (input_ids, attention_mask, …)
        # La etiqueta (labels)
        # Esto es lo que el Trainer usará en cada batch de entrenamiento

    def __len__(self):      # para usar el método len() --> da el numero total de ejemplos del dataset basándose en que se asume que hay una etiqueta por ejemplo
        return len(self.labels)
    # Esto permite que PyTorch sepa hasta qué índice se puede pedir, y cuántos batches se pueden crear

train_dataset = MyDataset(train_tokenized, y_train)
val_dataset = MyDataset(val_tokenized, y_val)
test_dataset = MyDataset(test_tokenized, y_test)

### f) Definir métricas de avaliação

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def classification_metrics(pred):   # El parámetro pred es un objeto especial (EvalPrediction) que contiene:  pred.label_ids → etiquetas reales, salidas crudas del modelo (logits)
    labels = pred.label_ids             # labels será un array como [1, 0, 1, 1, 0, ...]
    preds = pred.predictions.argmax(-1)       # pred.predictions es un array 2D con los logits de cada clase: algo como [(logit_clase_0, logit_clase_1), (logit_clase_0, logit_clase_1), ...] con argmax(-1) toma el índice de la clase con mayor valor --> [(0.4, 4.5), (3, 2.9)] entonces se quedará como [1, 0] clase 1, clase 0
    return {
        'accuracy': accuracy_score(labels, preds),
        'precision': precision_score(labels, preds, average='weighted'),
        'recall': recall_score(labels, preds, average='weighted'),
        'f1': f1_score(labels, preds, average='weighted')
    }

### g) Instalar otimizador e preparar o treino

In [ ]:
!pip install torch_optimizer

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.9/55.9 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 2.3 MB/s eta 0:00:00


### h) Treinar o modelo

In [ ]:
from transformers import DebertaForSequenceClassification, Trainer, TrainingArguments
# DebertaForSequenceClassification → modelo DeBERTa adaptado para clasificación de secuencias (texto → clase).
# Trainer → clase que encapsula el bucle de entrenamiento (batches, optimizador, evaluación, etc.).
# TrainingArguments → objeto para configurar todos los parámetros de entrenamiento (épocas, batch size, logs, etc.).


training_args = TrainingArguments(
    output_dir='./results',         # carpeta donde guardar: modelo entrenado, checkpoints intermedios, logs
    num_train_epochs=3,           # Una época = el modelo ha visto todos los ejemplos de entrenamiento una vez --> con 3 entonces se recorre el dataset 3 veces
    per_device_train_batch_size=16,     # Tamaño del batch de entrenamiento por dispositivo (CPU/GPU)
    per_device_eval_batch_size=64,      # Tamaño del batch durante la evaluación (validación/test)
    warmup_steps=500,       # Número de pasos de calentamiento (warmup) para la tasa de aprendizaje (learning rate) --> Al principio del entrenamiento, la tasa de aprendizaje se incrementa poco a poco durante 500 pasos en lugar de empezar con un valor alto de golpe --> esto ayuda a estabilizar el entrenamiento
    weight_decay=0.01,      # Parámetro de regularización L2 --> penaliza pesos demasiados grandes para no depender de ellos --> evitar sobreajuste
    logging_dir='./logs',           # Carpeta donde se guardan los logs de entrenamiento
    logging_steps=25,             # Cada 25 pasos de entrenamiento, el Trainer registrará métricas
    report_to="none"          # Indica que no se envíen logs a herramientas externas tipo TensorBoard, WandB, etc --> Solo mostrará en consola
)

# ¿Por qué es diferente del batch de entrenamiento?
# ✔️ En entrenamiento se calculan gradientes → se necesita más memoria
# ✔️ En evaluación NO se calculan gradientes → memoria libre
# Batch pequeño (4–16) --> Menos memoria consumida y mejor generalización, pero más lento
# Batch grande (64–512) --> Necesita menos pasos, Pero puede generalizar peor y consume más memoria CPU/GPU

# CONCLUSIÓN: el dataset termina cuando de 3 pasadas completas, las primeras 500 pasadas no se toman en cuenta porque sirven para ajustar el learning rate, se usan batch size de 16 datos para
#             el train y 64 para el calculo de las metricas (para que la validacion de las métricas sean más rápidas, se dé en menos pasos), regularización definida sobre 0.01 para ajustar el
#             sobreajuste y cada 25 pasos se hace la validación de las métricas

# Muchos estudiantes creen que “los primeros 500 batches no se usan”, pero eso es incorrecto --> Durante los primeros warmup_steps, la learning rate (LR) va subiendo progresivamente desde 0 hasta el valor máximo.
#                   --> Después continúa normalmente con el scheduler --> Todos esos batch sí actualizan pesos, pero lo hacen usando LR muy pequeña --> Evita que el modelo “explote” al principio
#                   --> Los modelos gigantes son muy inestables en los primeros pasos --> Si empiezas con una LR alta → el entrenamiento puede divergir

model = DebertaForSequenceClassification.from_pretrained(           # Carga un modelo preentrenado, DebertaForSequenceClassification añade una capa de clasificación encima del encoder base
    "PORTULAN/albertina-100m-portuguese-ptpt-encoder")                  # Al usar .from_pretrained(...): se descargan pesos ya entrenados en lenguaje portugués, solo se ajustan con fine-tuning para la tarea de humor vs no humor

# ¿El fine-tuning se ajusta de forma automática?
# Internamente, durante el fine-tuning:
# 🔹 1. Se añaden capas de clasificación al final del modelo  --> La capa final (la de clasificación) empieza sin entrenar, con pesos aleatorios (random initialization).
# 🔹 2. Se decide qué partes se entrenan --> por defecto ✔️ Todas las capas del modelo se entrenan (es decir, no congelamos capas principales y solo dejamos las ultimas, esto es comun en datasets más grandes) --> Los pesos del encoder DeBERTa se ajustan ligeramente --> Y la capa de clasificación se entrena desde cero

# ¿No sería mejor congelar las capas principales y entrenar solo las últimas?
# 👉 Depende del objetivo y del tamaño del dataset.

# ¿Por qué en Transformers NO se suelen congelar las capas?
# Porque los Transformers (BERT, RoBERTa, DeBERTa…) están diseñados para que el fine-tuning sea full, es decir:
# ✔️ Ajustar todas las capas mejora el rendimiento incluso con datasets muy pequeños
# Esto es fundamental:
# Los modelos tipo BERT NO se entrenan desde cero al hacer fine-tuning.
# Ya tienen los pesos muy bien inicializados.
# El fine-tuning solo hace micro-ajustes.
# Por eso es seguro tocar todas las capas.

# ¿Cuándo SÍ se congelan capas?
# 🔸 A) Cuando el dataset es extremadamente pequeño --> si hay 100 ejemplos el riesgo de sobreajustar es muy alto
# 🔸 B) Cuando quieres reducir los tiempos de entrenamiento --> Menos capas → menos gradientes → más rápido
# 🔸 C) Cuando haces aprendizaje multitarea --> Congelas algunas y entrenas otras.
# 🔸 D) Cuando haces feature extraction, no fine-tuning  --> (Ejemplo: BERT como extractor de embeddings.)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=classification_metrics
)

trainer.train()
# Inicia el proceso de entrenamiento.
  # Internamente hace:
  # Divide train_dataset en batches.
  # Pasa cada batch por el modelo.
  # Calcula la pérdida (loss) comparando predicciones vs labels.
  # Calcula gradientes y actualiza pesos.
  # Cada cierto número de pasos, evalúa en eval_dataset y llama a classification_metrics.
  # Guarda checkpoints y logs según TrainingArguments.

config.json:   0%|          | 0.00/779 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/279M [00:00<?, ?B/s]

Some weights of DebertaForSequenceClassification were not initialized from the model checkpoint at PORTULAN/albertina-100m-portuguese-ptpt-encoder and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss
25,0.689200
50,0.649500
75,0.474700
100,0.321500
125,0.042300
150,0.001800
175,0.000900
200,0.019900
225,0.000900
250,0.000500


TrainOutput(global_step=588, training_loss=0.10048506315797567, metrics={'train_runtime': 209.5183, 'train_samples_per_second': 44.903, 'train_steps_per_second': 2.806, 'total_flos': 411262354669824.0, 'train_loss': 0.10048506315797567, 'epoch': 3.0})

In [ ]:
# Como podemos ver usando un batch_size para el train de 16 datos, se termina de dar 3 pasadas completas al dataset en el paso 588 con una pérdida del 0.1 respecto a la predicción y con las
# métricas dadas

### i) Avaliar e guardar o modelo

In [ ]:
# Avaliação final no conjunto de teste --> se evalúa el modelo en el conjunto de test con los pesos ya entrenados en el step final (588)
trainer.evaluate(eval_dataset=test_dataset)

# ====================IMPORTANTE DEDUCCIÓN:==================================================================================================
# Es como lo que hicimos en el Milestone2 pero en este caso trasladado a grandes redes neuronales como en DL --> testeamos cada modelo
# En Milestone2 entrenamos de golpe (sin usar batch de datos) distintos modelos con distintos algoritmos y testeamos en el conjunto de validacion para ver cual era el mejor modelo a priori.
#               Despues testabmos en test el mejor solo. A parte, haciamos un GreadSearchCV que entrenaba varias veces el mismo modelo con distintos hiperparámetros y usaba su propio conjunto de validación interno (Cross Validation)
# En DL y en esta práctica usamos el conjunto de validación para testear como generaliza el modelo en cada batch de datos hasta completar una epoch (una pasada entera al dataset)
#           Sin embargo, no estamos comprobando los mejores hiperparámetros para el modelo como hacíamos con GridResearchCV en Milestone2. Luego testeamos el modelo final ya entrenado en el conjunto de test
#           Se usa el modelo de entrenar en batchs y testear en validación para ver de forma inmediata si el modelo está sobreajustandose o no y evitar un largo entrenamiento si vemos que el modelo se sobreajusta (pararlo antes)
#           En conclusión, el modelo de validación se usa para: 1. Medir rendimiento mientras se entrena y Ajustar hiperparámetros (pero en este ejercicio no lo hacemos)
# =======================================================================================================================================

# Guardar o modelo treinado
trainer.save_model()